# Pipeline Analysis

This notebook runs the classical preprocessing pipeline with mRNA enrichment enabled and checks whether the resulting feature matrix is ready for gene-split Random Forest experiments.

In [22]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.pipeline import SiRNADataPipeline

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Choose Data Source

In [23]:
from utils.merge_historic_data import load_merged_dataset

use_merged_dataset = True
historic_path = os.environ.get("CMSIRNA_RAW_HISTORIC_DATA_PATH")


cmsirna_path = os.environ.get("CMSIRNA_RAW_DATA_PATH")
assert cmsirna_path, "CMSIRNA_RAW_DATA_PATH is not set"

if use_merged_dataset:
    assert historic_path is not None, "Set historic_path before running merged mode"
    raw_df = load_merged_dataset(str(cmsirna_path), str(historic_path))
    data_source_label = "CMsiRNA + historic"
else:
    raw_df = pd.read_csv(cmsirna_path, sep="\t", low_memory=False)
    data_source_label = "CMsiRNA only"

if "Target_Gene" in raw_df.columns and "gene_target_symbol_name" not in raw_df.columns:
    raw_df = raw_df.rename(columns={"Target_Gene": "gene_target_symbol_name"})

required_columns = [
    "gene_target_symbol_name",
    "Sense_seqence",
    "Antisense_seqence",
    "Modification_Types_Sense_strand",
    "Modification_Types_Antisense_strand",
    "Cell_Type",
    "Concentration",
    "Time_of_administration",
    "Inhibition",
]
missing = [col for col in required_columns if col not in raw_df.columns]
assert not missing, f"Missing required raw-data columns: {missing}"

print("Data source:", data_source_label)
print("Raw shape:", raw_df.shape)
print("Unique genes in raw data:", raw_df["gene_target_symbol_name"].nunique())
raw_df.head()

loaded 3515 historic rows
merged 43153 CMsiRNA and 3515 historic rows into 46668
Data source: CMsiRNA + historic
Raw shape: (46668, 27)
Unique genes in raw data: 54


,ID,patent_ID,Authorization_status,Accession_number,gene_target_symbol_name,Gene_ID,The_name_of_double_helix,Antisense_seqence,length_anti_sense_strand,Modifications_AntiSense_strand,Modification_Types_Antisense_strand,Modification_locations_Antisense_strand,position_Antisense_strand,Sense_seqence,length_sense_strand,Modifications_sense_strand,Modification_Types_Sense_strand,Modification_locations_Sense_strand,position_Sense_strand,Inhibition,SD,Cell_Type,Concentration,Time_of_administration,Title,Modifications_AntiSense_strand_3_5,mRNA
0,001-01-01-00001-100n-48h-88.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D1,UUAGAAAGCUGAUGGACCAUAACUG,25.0,UUAGAAAGCUGAUGGACCAUAACUG,1*U || 2*U || 3*A || 4*G || 5*A || 6*A || 7*A ...,NaN,NaN,CAGUUAUGGUCCAUCAGCUUUCUAA,25.0,mCmAmGmUmUmAmUGGUCCAUCAGC(mU)mUmUmCmUmAmA,1*2'-O-Methylcytidine || 2*2'-O-Methyladenosin...,Sugar,"1,2,3,4,5,6,7,19,20,21,22,23,24,25",88.0,NaN,Hela,100nM,48h,用于抑制CTNNB1靶基因mRNA表达的寡核酸分子及其成套组合物,GUCAAUACCAGGUAGUCGAAAGAUU,NaN
1,001-01-01-00002-100n-48h-90.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D2,UACAAUAGCAGACACCAUCUGAGGA,25.0,UACAAUAGCAGACACCAUCUGAGGA,1*U || 2*A || 3*C || 4*A || 5*A || 6*U || 7*A ...,NaN,NaN,UCCUCAGAUGGUGUCUGCUAUUGUA,25.0,mUmCmCmUmCmAmGAUGGUGUCUGC(mU)mAmUmUmGmUmA,1*2'-O-Methyluridine || 2*2'-O-Methylcytidine ...,Sugar,"1,2,3,4,5,6,7,19,20,21,22,23,24,25",90.0,NaN,Hela,100nM,48h,用于抑制CTNNB1靶基因mRNA表达的寡核酸分子及其成套组合物,AGGAGUCUACCACAGACGAUAACAU,NaN
2,001-01-01-00003-100n-48h-90.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D3,UGAACAAGACGUUGACUUGGAUCUG,25.0,UGAACAAGACGUUGACUUGGAUCUG,1*U || 2*G || 3*A || 4*A || 5*C || 6*A || 7*A ...,NaN,NaN,CAGAUCCAAGUCAACGUCUUGUUCA,25.0,mCmAmGmAmUmCmCAAGUCAACGUC(mU)mUmGmUmUmCmA,1*2'-O-Methylcytidine || 2*2'-O-Methyladenosin...,Sugar,"1,2,3,4,5,6,7,19,20,21,22,23,24,25",90.0,NaN,Hela,100nM,48h,用于抑制CTNNB1靶基因mRNA表达的寡核酸分子及其成套组合物,GUCUAGGUUCAGUUGCAGAACAAGU,NaN
3,001-01-01-00004-100n-48h-89.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D4,UUUAGUUGCAGCAUCUGAAAGAUUC,25.0,UUUAGUUGCAGCAUCUGAAAGAUUC,1*U || 2*U || 3*U || 4*A || 5*G || 6*U || 7*U ...,NaN,NaN,GAAUCUUUCAGAUGCUGCAACUAAA,25.0,mGmAmAmUmCmUmUUCAGAUGCUGC(mA)mAmCmUmAmAmA,1*2'-O-Methylguanosine || 2*2'-O-Methyladenosi...,Sugar,"1,2,3,4,5,6,7,19,20,21,22,23,24,25",89.0,NaN,Hela,100nM,48h,用于抑制CTNNB1靶基因mRNA表达的寡核酸分子及其成套组合物,CUUAGAAAGUCUACGACGUUGAUUU,NaN
4,001-01-01-00005-100n-48h-87.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D5,UUUCGAAUCAAUCCAACAGUAGCCU,25.0,UUUCGAAUCAAUCCAACAGUAGCCU,1*U || 2*U || 3*U || 4*C || 5*G || 6*A || 7*A ...,NaN,NaN,AGGCUACUGUUGGAUUGAUUCGAAA,25.0,mAmGmGmCmUmAmCUGUUGGAUUGA(mU)mUmCmGmAmAmA,1*2'-O-Methyladenosine || 2*2'-O-Methylguanosi...,Sugar,"1,2,3,4,5,6,7,19,20,21,22,23,24,25",87.0,NaN,Hela,100nM,48h,用于抑制CTNNB1靶基因mRNA表达的寡核酸分子及其成套组合物,UCCGAUGACAACCUAACUAAGCUUU,NaN


## Run Full Pipeline With mRNA

In [24]:
sirna_pipeline = SiRNADataPipeline(target_len=25, fetch_missing_mrna=True)

enriched_df = sirna_pipeline.enrich_dataset_with_encodings(
    raw_df,
    strict_cleaning=True,
    add_mrna=True,
)



print("Enriched shape:", enriched_df.shape)
enriched_df.head()

Running qc and data cleaning
dropped 4233 rows for in-vivo readings
dropped 565 rows for mM readings
dropped 115 rows for unknown or unwanted cell lines
dropped 47 rows for out-of-range inhibition
dropped 1749 rows for missing or unknown concentration
dropped 796 rows for concentration > 200 nM
filled 7522 rows for missing time of administration
dropped 2198 rows with a missing or >25 nt strand
dropped 6 columns: ['Modification_locations_Sense_strand', 'Modification_locations_Antisense_strand', 'Modifications_sense_strand', 'Modifications_AntiSense_strand_3_5', 'position_Antisense_strand', 'position_Sense_strand']
Mapping mRNA structural profiles
Error reading reference dataset: [Errno 2] No such file or directory: '/home/larsena8/software/fennec/src/fennec/support_files/train_data_v1.1.0_N=27742.csv'
Loaded 13 gene sequences from local cache
Loaded 0 gene sequences from reference CSV
Building gene -> mRNA mapping for 54 genes...
[1/54] Processing CTNNB1...
  Found in local cache (3074

,ID,patent_ID,Authorization_status,Accession_number,gene_target_symbol_name,Gene_ID,The_name_of_double_helix,Antisense_seqence,length_anti_sense_strand,Modifications_AntiSense_strand,Modification_Types_Antisense_strand,Sense_seqence,length_sense_strand,Modification_Types_Sense_strand,Inhibition,SD,Cell_Type,Concentration,Time_of_administration,Title,mRNA,Concentration_nM,Time_of_administration_h,mRNA_five_prime,mRNA_three_prime,edit_distance,target_site_pct,Sense_Sequence_One_Hot,Antisense_Sequence_One_Hot,Sense_Acid_One_Hot,Sense_Sugar_One_Hot,Sense_Linker_One_Hot,Antisense_Acid_One_Hot,Antisense_Sugar_One_Hot,Antisense_Linker_One_Hot,Cell_Type_One_Hot,Concentration_log10_nM,Concentration_norm,Time_norm
0,001-01-01-00001-100n-48h-88.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D1,UUAGAAAGCUGAUGGACCAUAACUG,25.0,UUAGAAAGCUGAUGGACCAUAACUG,1*U || 2*U || 3*A || 4*G || 5*A || 6*A || 7*A ...,CAGUUAUGGUCCAUCAGCUUUCUAA,25.0,1*2'-O-Methylcytidine || 2*2'-O-Methyladenosin...,88.0,NaN,Hela,100.0,48.0,用于抑制CTNNB1靶基因mRNA表达的寡核酸分子及其成套组合物,AAGCCUCUCGGUCUGUGGCAGCAGCGUUGGCCCGGCCCCGGGAGCG...,100.0,48.0,AAGCCUCUCGGUCUGUGGCAGCAGCGUUGGCCCGGCCCCGGGAGCG...,CACACUAACCAAGCUGAGUUUCCUAUGGGAACAAUUGAAGUAAACU...,0.0,0.236825,"[[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [1.0, 0....","[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...",2.0,0.950412,0.268293
1,001-01-01-00002-100n-48h-90.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D2,UACAAUAGCAGACACCAUCUGAGGA,25.0,UACAAUAGCAGACACCAUCUGAGGA,1*U || 2*A || 3*C || 4*A || 5*A || 6*U || 7*A ...,UCCUCAGAUGGUGUCUGCUAUUGUA,25.0,1*2'-O-Methyluridine || 2*2'-O-Methylcytidine ...,90.0,NaN,Hela,100.0,48.0,用于抑制CTNNB1靶基因mRNA表达的寡核酸分子及其成套组合物,AAGCCUCUCGGUCUGUGGCAGCAGCGUUGGCCCGGCCCCGGGAGCG...,100.0,48.0,AAGCCUCUCGGUCUGUGGCAGCAGCGUUGGCCCGGCCCCGGGAGCG...,CACACUAACCAAGCUGAGUUUCCUAUGGGAACAAUUGAAGUAAACU...,0.0,0.255693,"[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0....","[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...",2.0,0.950412,0.268293
2,001-01-01-00003-100n-48h-90.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D3,UGAACAAGACGUUGACUUGGAUCUG,25.0,UGAACAAGACGUUGACUUGGAUCUG,1*U || 2*G || 3*A || 4*A || 5*C || 6*A || 7*A ...,CAGAUCCAAGUCAACGUCUUGUUCA,25.0,1*2'-O-Methylcytidine || 2*2'-O-Methyladenosin...,90.0,NaN,Hela,100.0,48.0,用于抑制CTNNB1靶基因mRNA表达的寡核酸分子及其成套组合物,AAGCCUCUCGGUCUGUGGCAGCAGCGUUGGCCCGGCCCCGGGAGCG...,100.0,48.0,AAGCCUCUCGGUCUGUGGCAGCAGCGUUGGCCCGGCCCCGGGAGCG...,CACACUAACCAAGCUGAGUUUCCUAUGGGAACAAUUGAAGUAAACU...,0.0,0.431034,"[[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0], [1.0, 0....","[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]...","[[1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ...",2.0,0.950412,0.268293
3,001-01-01-00004-100n-48h-89.00,CN107365771B,Unknown Status,NM_001330729.2,CTNNB1,1499.0,D4,UUUAGUUGCAGCAUCUGAAAGAUUC,25.0,UUUAGUUGCAGCAUCUGAAAGAUUC,1*U || 2*U || 3*U || 4*A || 5*G || 6*U || 7*U ...,GAAUCUUUCAGAUGCUGCAACUAAA,25.0,1*2'-O-Methylguanosine || 2*2'-O-Methyladenosi

## Check mRNA And Alignment Coverage

In [25]:
coverage_summary = pd.Series({
    "rows": len(enriched_df),
    "unique_genes": enriched_df["gene_target_symbol_name"].nunique(),
    "rows_with_mRNA": int(enriched_df["mRNA"].notna().sum()),
    "rows_with_5p_utr": int(enriched_df["mRNA_five_prime"].notna().sum()),
    "rows_with_3p_utr": int(enriched_df["mRNA_three_prime"].notna().sum()),
    "rows_with_edit_distance": int(enriched_df["edit_distance"].notna().sum()),
    "rows_with_target_site_pct": int(enriched_df["target_site_pct"].notna().sum()),
})
coverage_summary

rows                         36965
unique_genes                    54
rows_with_mRNA               36965
rows_with_5p_utr             35919
rows_with_3p_utr             35919
rows_with_edit_distance      36965
rows_with_target_site_pct    36965
dtype: int64

In [26]:
alignment_cols = ["gene_target_symbol_name", "Antisense_seqence", "mRNA", "edit_distance", "target_site_pct"]
enriched_df[alignment_cols].sample(min(5, len(enriched_df)), random_state=42)

,gene_target_symbol_name,Antisense_seqence,mRNA,edit_distance,target_site_pct
42632,PCSK9,CAGAGUGAGUGAGUUCCAGGC,AGCGACGUCGAGGCGCUCAUGGUUGCAGGCGGGCGCCGCCGUUCAG...,0.0,0.649554
11950,PNPLA3,AGUUAGGUGAAAAAGGUGUUCUA,AGAGAGCGCUUGCGGGCGCCGGGCGGAGCUGCUGCGGAUCAGGACC...,0.0,0.765347
30527,HSD17B13,UGACAUGAGGUUUUGAUACUU,ACACAAGGACUGAACCAGAAGGAAGAGGACAGAGCAAAGCCAUGAA...,3.0,0.247345
26403,HSD17B13,ACAACGACUCCAAGUAGGAGUAG,ACACAAGGACUGAACCAGAAGGAAGAGGACAGAGCAAAGCCAUGAA...,1.0,0.037611
1731,INHBE,ACCATUCCCUGGAGCCACACUCC,AGUAGCCAGACAUGAGCUGUGAGGGUCAAGCACAGCUAUCCAUCAG...,1.0,0.192683


## Build Classical ML Matrix

In [27]:
X, groups, y = sirna_pipeline.prepare_for_classical_ml(enriched_df, target_column="Inhibition", use_normalized_conditions=False)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique groups:", len(np.unique(groups)))

Feature matrix X shape: (36965, 1425), target y shape: (36965,) 
X shape: (36965, 1425)
y shape: (36965,)
Unique groups: 54


## Readiness Checks For Random Forest

In [28]:
readiness = pd.Series({
    "X_total_nan": int(np.isnan(X).sum()),
    "y_total_nan": int(np.isnan(y).sum()),
    "rows_with_any_nan_in_X": int(np.isnan(X).any(axis=1).sum()),
    "rows_with_nan_y": int(np.isnan(y).sum()),
    "rows_with_empty_group": int(pd.isna(groups).sum()),
})
readiness

X_total_nan               0
y_total_nan               0
rows_with_any_nan_in_X    0
rows_with_nan_y           0
rows_with_empty_group     0
dtype: int64

In [29]:
nan_rows = np.where(np.isnan(X).any(axis=1))[0]
if len(nan_rows) == 0:
    print("No NaNs found in X. The matrix is ready for a RandomForestRegressor baseline.")
else:
    print(f"Found {len(nan_rows)} rows with NaNs in X. Inspecting the first few rows:")
    display(enriched_df.iloc[nan_rows[:5]][[
        "gene_target_symbol_name", "Concentration", "Time_of_administration",
        "Concentration_nM", "Time_of_administration_h", "Concentration_norm", "Time_norm",
        "mRNA", "edit_distance", "target_site_pct", "Inhibition"
    ]])

No NaNs found in X. The matrix is ready for a RandomForestRegressor baseline.


## Optional: Save The Enriched Dataset

In [30]:
processed_dir = os.environ.get("CMSIRNA_PROCESSED_DIR")
if processed_dir:
    output_path = Path(processed_dir) / "pipeline_analysis_enriched.csv"
    enriched_df.to_csv(output_path, index=False)
    print("Saved enriched dataframe to:", output_path)
else:
    print("CMSIRNA_PROCESSED_DIR is not set; skipping CSV export.")

Saved enriched dataframe to: /Users/yesminemaalej/siRNA-Pharmaceutical-Modeling-with-Foundation-Models/data/processed/pipeline_analysis_enriched.csv
